# Phishing Detection: Deploying the Detector
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sjcALane/llm-security-labs/blob/main/notebooks/06_deployment.ipynb)

A model in a notebook isn't a product. We put the detector behind a web API with basic security controls, then measure how fast it is.

In [ ]:
# In Colab: Runtime > Change runtime type > T4 GPU. Then run this cell.
try:
    import seclab
except ImportError:
    !pip -q install git+https://github.com/sjcALane/llm-security-labs
    import seclab

# Optional: keep saved adapters between Colab sessions.
# from google.colab import drive
# drive.mount("/content/drive")

from seclab import attacks, config, data, lora, metrics, notebook
from seclab.model import load, score
notebook.banner()

In [ ]:
df = data.load_emails()
train, val, test = data.split(df, seed=config.SEED)
test_small = data.sample(test, config.N_TEST, seed=config.SEED)
y = test_small["label"].to_numpy()
emails = test_small["text"].tolist()
print(f"{len(train)} training emails, {len(test_small)} test emails")

In [ ]:
lm, tok = load(config.MODEL_NAME, config.device())
train_small = data.sample(train, config.N_TRAIN, seed=config.SEED)
ft = lora.load_or_train(
    lm, tok, train_small, notebook.workdir() / "adapter-clean",
    batch_size=config.BATCH, micro_batch=config.MICRO_BATCH,
)

## Merge the adapter

For serving we fold the adapter into the base weights once:

$$W_{\text{merged}} = W + \frac{\alpha}{r}\,BA$$

After that the model has the same shape as the original, so it runs exactly as fast.

In [ ]:
import numpy as np

check = emails[:16]
p_before = score(ft, tok, check)
merged = ft.merge_and_unload()
p_after = score(merged, tok, check)
print("largest change in p(phishing):", np.abs(p_before - p_after).max())

**Question 1.** The change is tiny but not zero. Why?

**Your answer:**

## The service

`seclab.serve.create_app` wraps any scoring function in a web API with these controls:

| Control | What it stops |
|---|---|
| Inputs over 5,000 characters are rejected | Huge requests that tie up the GPU |
| `normalize()` runs before scoring | Homoglyph and zero-width evasion |
| Rate limit per client | An attacker testing thousands of versions of an email to find one that gets through |
| Logs store a hash of the email, not the email | Private email showing up in log files |

In [ ]:
import threading
import time

import httpx
import uvicorn
from seclab.serve import create_app


def score_fn(texts):
    return score(merged, tok, list(texts))


app = create_app(score_fn, model_version="phish-lora-v1", rate_limit=600)
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started:
    time.sleep(0.1)

api = httpx.Client(base_url="http://127.0.0.1:8000", timeout=60)
api.get("/health").json()

**Python note.** The server runs in a background thread, so this notebook can act as its client.

In [ ]:
phish = test_small[test_small["label"] == 1]["text"].iloc[0]
safe = test_small[test_small["label"] == 0]["text"].iloc[0]
for text in [phish, safe, attacks.homoglyph(phish)]:
    print(api.post("/classify", json={"text": text}).json())

In [ ]:
r = api.post("/classify", json={"text": "x" * 6000})
print(r.status_code, r.json()["detail"][0]["msg"])

The rate limit, shown on a second copy of the app that allows 3 requests a minute:

In [ ]:
from fastapi.testclient import TestClient

strict = TestClient(create_app(score_fn, rate_limit=3))
print([strict.post("/classify", json={"text": safe}).status_code for _ in range(5)])

## How fast is it?

In [ ]:
sample = emails[: config.size(100, 8)]
lat = []
start = time.perf_counter()
for text in sample:
    t0 = time.perf_counter()
    api.post("/classify", json={"text": text})
    lat.append(time.perf_counter() - t0)
api_rate = len(sample) / (time.perf_counter() - start)
print(f"API, one email per request: p50 {1000 * np.median(lat):.0f} ms, "
      f"p95 {1000 * np.percentile(lat, 95):.0f} ms, {api_rate:.1f} emails/s")

start = time.perf_counter()
score(merged, tok, sample, batch_size=32)
batch_rate = len(sample) / (time.perf_counter() - start)
print(f"direct, batches of 32: {batch_rate:.1f} emails/s")

**Question 2.** Batched scoring handles far more emails per second than the API. Why? How could the service get some of that speed back?

**Your answer:**

**Question 3.** A bank receives 2 million emails a day. Using your numbers, how many GPUs like this one would it need? Show your work and state your assumptions.

**Your answer:**

## Threat model

The OWASP Top 10 for LLM Applications (2025) lists the most common security risks in LLM systems.

**Question 4.** For each risk below, write one sentence on how it applies to our detector and name one control.

1. LLM01 Prompt Injection
2. LLM02 Sensitive Information Disclosure
3. LLM03 Supply Chain (where did our base model come from?)
4. LLM04 Data and Model Poisoning
5. LLM10 Unbounded Consumption

**Your answer:**

In [ ]:
server.should_exit = True